[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Transactions


## What you will be able to do

Group changes into a transaction that `commit` saves and `rollback` throws away, completely or not at
all, and know when sqlite3 has opened one for you. Write a group of changes as a `with conn:` block
that commits when it succeeds and rolls back when it raises, say what another connection sees before
and after a commit, and undo part of a transaction with a savepoint. Recognize the changes lost to a
missing commit, and the ones saved by a commit that nobody meant to include them in.


## The idea

### The problem

Svalbard sent no readings on 2 March, and the network decides to fill the day with estimates: delete
the day's 24 empty readings, then insert 24 estimated ones. Those 25 statements belong together. If
the program stops partway through, because one estimate cannot be worked out or the laptop's battery
runs out, the file is left with neither version of the day: the empty readings deleted, and only
some of the estimates in their place. Nothing about the file would say that anything went wrong.

A transaction makes the 25 statements one change, which happens completely or not at all. And
sqlite3 has been opening transactions all along: every insert in this guide went into one, which is
why the notebooks called `commit`. Leave that call out and the rows look saved, since the connection
that wrote them reads them back, until the file is opened again and they are gone.

### What a transaction is

> A **transaction** is a group of changes that the database applies completely or not at all. After
> it begins, **`commit`** makes every change in it permanent and visible to other connections at
> once, and **`rollback`** throws every change in it away, leaving the database as the transaction
> found it. Until one of them runs, the changes exist only for the connection that made them. sqlite3
> opens a transaction for you before an `INSERT`, `UPDATE`, `DELETE` or `REPLACE` when none is open,
> and `conn.in_transaction` reports whether one is. A **savepoint** marks a place inside a
> transaction, so that `ROLLBACK TO` can undo the changes after the mark and keep the ones before it.

### Why it works that way

- **All or nothing survives a crash.** SQLite's documentation says every change within a transaction
  either occurs completely or not at all, even when writing it out is interrupted by a program crash,
  an operating system crash or a power failure.
- **A transaction stays open until something ends it.** sqlite3 commits nothing on its own: not
  after a statement, not when a notebook cell ends, and not when the connection closes, since
  `close()` throws an open transaction's changes away.
- **Other connections see committed changes only.** Until the commit, every other connection reads
  the database as it was, so no reader ever sees half of a change.
- **`with conn:` ends a transaction, and does nothing else.** Leaving the block commits if nothing
  raised and rolls back if something did. It does not open a transaction, and it does not close the
  connection.
- **Transactions do not nest.** `BEGIN` inside an open transaction is an error, and a `with conn:`
  inside another commits or rolls back everything the outer block has done so far. A savepoint is
  the way to mark a step inside a transaction.
- **Some statements commit what is pending before they run.** `executescript` commits an open
  transaction first, so a later `rollback` finds nothing to undo.

### Where you will meet this

Every relational database works in transactions, and every library that sits on one passes them on.
The **Connections and Transactions** notebook in the **SQLAlchemy, Deep Dive** guide meets the same
unsaved rows through an engine, and that guide's **Testing a Data Layer** notebook rolls back a
transaction after every test, so each test starts clean. The **Transactions** notebook in the
**Peewee, Deep Dive** guide wraps transactions and savepoints in one `atomic()` block that nests.
PostgreSQL, in the **Transactions and Errors** notebook of the **asyncpg and psycopg3, Deep Dive**
guide, refuses every statement after an error until the transaction is rolled back, where SQLite
carries on. In this guide, the **autocommit and isolation_level** notebook sets out exactly when
sqlite3 opens a transaction and how to change that, and the **Concurrency and WAL** notebook shows
what an open transaction locks.

### What this notebook covers

- `commit`, and what another connection sees before and after it
- `rollback`, and `in_transaction`
- `with conn:`, which commits a block that finishes and rolls back one that raises
- All or nothing: replacing a day of readings in one transaction
- Savepoints, for undoing part of a transaction
- When to use `with conn:`, `commit()` by hand, or a savepoint
- A night's load of several stations in one transaction, with a savepoint for every station
- Six errors: a connection closed before its commit, `BEGIN` inside an open transaction, a commit
  hidden in `executescript`, an exception caught halfway through a change, a `with conn:` inside
  another, and `COMMIT` with no transaction open

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "readings.db"
    conn = sqlite3.connect(path)
    other = sqlite3.connect(path)
    conn.execute("CREATE TABLE readings (hour TEXT, celsius REAL)")

    conn.execute("INSERT INTO readings VALUES ('2025-12-02T00:00', -7.0)")
    count = "SELECT COUNT(*) FROM readings"
    print("in a transaction:", conn.in_transaction)
    print("this connection sees", conn.execute(count).fetchone()[0],
          "and the other sees", other.execute(count).fetchone()[0])

    conn.commit()
    print("after commit, the other sees", other.execute(count).fetchone()[0])

    conn.execute("INSERT INTO readings VALUES ('2025-12-02T01:00', -7.2)")
    conn.close()
    print("after closing without a commit, the other sees", other.execute(count).fetchone()[0])
    other.close()
```

```
in a transaction: True
this connection sees 1 and the other sees 0
after commit, the other sees 1
after closing without a commit, the other sees 1
```

Two connections to one file. The insert opened a transaction, so the connection that made it could
read its row while the other connection could not. `commit` made the row permanent and visible to
both. The second insert opened another transaction, and closing the connection without a commit threw
that row away, so the other connection never saw it and never will.


## Setup

Five imports, and the stations' year, built into the two tables the **Tables and Queries** notebook
designed.

- `sqlite3` builds the database and runs every statement
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook
  made, which also restores a day of readings in Common errors
- `Path` names the scratch folder and the database in it
- `shutil` removes the scratch folder at the end


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


## Worked examples

### commit, and what another connection sees

Two connections to the same file make a transaction visible: `writer` changes the database, and
`reader` stands for every other program that opens it. `count_readings` asks one connection how many
readings a station has on a day, as that connection sees the database, and `INSERT_READING` is the
insert the rest of the notebook uses:


In [2]:
writer = sqlite3.connect(DATABASE)
reader = sqlite3.connect(DATABASE)
station_ids = dict(writer.execute("SELECT name, id FROM stations"))
INSERT_READING = "INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)"


def count_readings(conn, station, day):
    """How many readings a station has on a day, as this connection sees the database."""
    return conn.execute("SELECT COUNT(*) FROM readings WHERE station_id = ? AND hour LIKE ?",
                        (station_ids[station], f"{day}%")).fetchone()[0]


print("in a transaction before the insert:", writer.in_transaction)
writer.executemany(INSERT_READING,
                   [(station_ids["Kirkenes"], f"2025-12-02T{hour:02d}:00", celsius)
                    for hour, celsius in enumerate([-7.0, -7.2, -7.5])])
print("in a transaction after the insert: ", writer.in_transaction)
print("the writer sees", count_readings(writer, "Kirkenes", "2025-12-02"),
      "and the reader sees", count_readings(reader, "Kirkenes", "2025-12-02"))

writer.commit()
print("in a transaction after commit:     ", writer.in_transaction)
print("the writer sees", count_readings(writer, "Kirkenes", "2025-12-02"),
      "and the reader sees", count_readings(reader, "Kirkenes", "2025-12-02"))


in a transaction before the insert: False
in a transaction after the insert:  True
the writer sees 3 and the reader sees 0
in a transaction after commit:      False
the writer sees 3 and the reader sees 3


sqlite3 opened the transaction itself, just before the insert, and `in_transaction` reported it.
Until the commit, the rows existed for `writer` and for no one else, so `reader` counted none. The
commit made all three visible at once, and ended the transaction. A query opens no transaction, and
the **autocommit and isolation_level** notebook sets out exactly which statements do.

### rollback, and in_transaction

`rollback` throws away every change since the transaction began. Here a delete meant for one hour of
Bergen's midsummer day takes the whole day, and `rollback` puts it back:


In [3]:
writer.execute("DELETE FROM readings WHERE station_id = ? AND hour LIKE ?", (station_ids["Bergen"], "2025-06-21%"))
print("after the delete:  ", writer.in_transaction, count_readings(writer, "Bergen", "2025-06-21"), "readings")

writer.rollback()
print("after the rollback:", writer.in_transaction, count_readings(writer, "Bergen", "2025-06-21"), "readings")

writer.rollback()
writer.commit()
print("a rollback and a commit with nothing open:", writer.in_transaction, count_readings(writer, "Bergen", "2025-06-21"))


after the delete:   True 0 readings
after the rollback: False 24 readings
a rollback and a commit with nothing open: False 24


The delete ran, and even `writer` saw no readings left, but nothing was permanent until a commit, so
the rollback restored all 24 and closed the transaction. `commit()` and `rollback()` with no
transaction open do nothing at all, so calling either one again is harmless.

### with conn:

A connection used in a `with` block commits when the block finishes and rolls back when the block
raises, and the exception still reaches the code outside. The first block below fails after its
insert, and the second succeeds:


In [4]:
try:
    with writer:
        writer.execute(INSERT_READING,
                       (station_ids["Kirkenes"], "2025-12-02T03:00", -7.7))
        raise ValueError("the next estimate could not be worked out")
except ValueError as error:
    print("raised:", error)
print("after the block that raised:  ", writer.in_transaction, count_readings(reader, "Kirkenes", "2025-12-02"))

with writer:
    writer.execute(INSERT_READING,
                   (station_ids["Kirkenes"], "2025-12-02T03:00", -7.7))
print("after the block that finished:", writer.in_transaction, count_readings(reader, "Kirkenes", "2025-12-02"))
print("the connection is still open: ", writer.execute("SELECT 1").fetchone())


raised: the next estimate could not be worked out
after the block that raised:   False 3
after the block that finished: False 4
the connection is still open:  (1,)


The block that raised left no row behind, and the one that finished committed its row, which `reader`
counted. Neither closed the connection. A `with conn:` block is the usual way to write a transaction,
since no path out of it forgets to commit or to roll back.

### All or nothing: replacing a day of readings

`fill_day` replaces a day's empty readings with estimates inside one `with` block, so the delete and
the 24 inserts succeed together or not at all. The first attempt uses estimates that arrived with one
hour missing, and the second works them out by drawing a straight line from the last reading before
the day to the first reading after it:


In [5]:
def fill_day(conn, station, day, estimate):
    """Replace a day's empty readings with estimates in one transaction: the whole day, or none of it."""
    with conn:
        conn.execute("DELETE FROM readings WHERE station_id = ? AND hour LIKE ? AND celsius IS NULL",
                     (station_ids[station], f"{day}%"))
        for hour in range(24):
            conn.execute(INSERT_READING,
                         (station_ids[station], f"{day}T{hour:02d}:00", estimate(hour)))


def empty_and_filled(conn, station, day):
    """How many of a day's readings are empty, and how many hold a value."""
    return conn.execute("""
        SELECT COUNT(*) - COUNT(celsius), COUNT(celsius) FROM readings WHERE station_id = ? AND hour LIKE ?
    """, (station_ids[station], f"{day}%")).fetchone()


last_before = writer.execute("SELECT celsius FROM readings WHERE station_id = ? AND hour = ?",
                             (station_ids["Svalbard"], "2025-03-01T23:00")).fetchone()[0]
first_after = writer.execute("SELECT celsius FROM readings WHERE station_id = ? AND hour = ?",
                             (station_ids["Svalbard"], "2025-03-03T00:00")).fetchone()[0]


def straight_line(hour):
    """An estimate on the straight line from the last reading before the day to the first one after."""
    return round(last_before + (first_after - last_before) * (hour + 1) / 25, 1)


arrived = {hour: straight_line(hour) for hour in range(24) if hour != 14}   # estimates with 14:00 missing

print("empty and filled before:", empty_and_filled(reader, "Svalbard", "2025-03-02"))
try:
    fill_day(writer, "Svalbard", "2025-03-02", lambda hour: arrived[hour])
except KeyError as error:
    print("no estimate for hour", error)
print("after the failed attempt:", empty_and_filled(reader, "Svalbard", "2025-03-02"))

fill_day(writer, "Svalbard", "2025-03-02", straight_line)
print("after the second attempt:", empty_and_filled(reader, "Svalbard", "2025-03-02"))


empty and filled before: (24, 0)
no estimate for hour 14
after the failed attempt: (24, 0)
after the second attempt: (0, 24)


The first attempt deleted the 24 empty readings and inserted 14 estimates before hour 14 raised
`KeyError`. The `with` block rolled all of that back, so the day was exactly as it had been, 24 empty
readings and no estimates, rather than 14 estimates and a hole. The second attempt finished, and the
day now holds 24 estimates.

### Savepoints

A savepoint marks a place inside a transaction. `ROLLBACK TO` undoes everything after the mark and
keeps the transaction open with the changes before it, and `RELEASE` removes the mark. A savepoint
needs a transaction around it, which `BEGIN` opens explicitly, and `BEGIN` works only while no
transaction is open:


In [6]:
print("in a transaction before BEGIN:", writer.in_transaction)
writer.execute("BEGIN")
writer.execute(INSERT_READING, (station_ids["Kirkenes"], "2025-12-02T04:00", -7.9))
writer.execute(INSERT_READING, (station_ids["Kirkenes"], "2025-12-02T05:00", -8.1))

writer.execute("SAVEPOINT before_six")
writer.execute(INSERT_READING, (station_ids["Kirkenes"], "2025-12-02T06:00", 81.0))
print("with the reading of 81.0:", count_readings(writer, "Kirkenes", "2025-12-02"), "readings")
writer.execute("ROLLBACK TO before_six")
writer.execute("RELEASE before_six")
print("after ROLLBACK TO:       ", count_readings(writer, "Kirkenes", "2025-12-02"), "readings")
print("still in a transaction:  ", writer.in_transaction)

writer.commit()
print("after commit, the reader sees", count_readings(reader, "Kirkenes", "2025-12-02"))


in a transaction before BEGIN: False
with the reading of 81.0: 7 readings
after ROLLBACK TO:        6 readings
still in a transaction:   True
after commit, the reader sees 6


`ROLLBACK TO` undid the impossible reading of 81.0 degrees and nothing before the savepoint, and the
transaction stayed open until the commit saved the 04:00 and 05:00 readings. Without the `BEGIN`, the
`SAVEPOINT` would have started a transaction of its own, and releasing that outermost savepoint would
have committed it on the spot.

### with conn:, commit() by hand, or a savepoint

A transaction can be ended three ways, and they suit different jobs:

| Write | When | Why |
|---|---|---|
| `with conn:` | a group of changes that must succeed or fail together | it commits on success and rolls back on any exception, so no path forgets either |
| `commit()` and `rollback()` by hand | code that chooses its own moment to save, such as a loader committing every thousand rows | the program decides when, and has to remember to |
| `SAVEPOINT`, `ROLLBACK TO` and `RELEASE` | a step inside a transaction that may fail without spoiling the rest | it undoes part of a transaction and keeps the rest of it open |

The default is a `with conn:` block for every group of changes, with savepoints inside it for the
steps that are allowed to fail, and never a `with conn:` block inside another, which one of the
Common errors shows.

### A night's load, one transaction with a savepoint for every station

The pieces of this notebook in one job: a night's readings arrive as text from several stations, and
they are loaded in one transaction, so the night appears to other connections all at once. A
savepoint around every station's rows lets one station's broken export be skipped without losing the
rest. `load_night` refuses to start while a transaction is already open, since its `BEGIN` could not
nest, and `reader` looks at the database before the block ends:


In [7]:
NIGHT = {
    "Kirkenes": [("2026-01-01T00:00", "-8.2"), ("2026-01-01T01:00", "-8.4")],
    "Tromso": [("2026-01-01T00:00", "-3.1"), ("2026-01-01T01:00", "--")],
    "Bergen": [("2026-01-01T00:00", "4.0"), ("2026-01-01T01:00", "4.2")],
}


def load_night(conn, exports):
    """Load every station's export in one transaction, skipping a station whose export will not parse."""
    if conn.in_transaction:
        raise RuntimeError("commit or roll back the open transaction first")
    loaded, skipped = [], []
    with conn:
        conn.execute("BEGIN")
        for station, rows in exports.items():
            conn.execute("SAVEPOINT station")
            try:
                for hour, text in rows:
                    conn.execute(INSERT_READING,
                                 (station_ids[station], hour, float(text)))
            except ValueError as error:
                conn.execute("ROLLBACK TO station")
                skipped.append((station, str(error)))
            else:
                loaded.append(station)
            conn.execute("RELEASE station")
        print("before the block ends, the reader sees", count_readings(reader, "Kirkenes", "2026-01-01"), "readings")
    return loaded, skipped


loaded, skipped = load_night(writer, NIGHT)
print("loaded:", loaded)
print("skipped:", skipped)
print("after the block, the reader sees:", {station: count_readings(reader, station, "2026-01-01") for station in NIGHT})


before the block ends, the reader sees 0 readings
loaded: ['Kirkenes', 'Bergen']
skipped: [('Tromso', "could not convert string to float: '--'")]
after the block, the reader sees: {'Kirkenes': 2, 'Tromso': 0, 'Bergen': 2}


Kirkenes and Bergen loaded, and Tromso's first reading went in and came back out with `ROLLBACK TO`
when its second would not parse. None of it was visible until the `with` block committed, and then
all of it was, four readings from two stations. An error other than a bad value, such as a station
missing from `station_ids`, would have left the block with an exception and rolled back the whole
night.

### Where each part came from

| In the load | What it relies on | The section that showed it |
|---|---|---|
| `with conn:` around the night | a block that commits when it finishes and rolls back when it raises | with conn: |
| `if conn.in_transaction:` | knowing whether a transaction is already open | rollback, and in_transaction |
| `conn.execute("BEGIN")` | an explicit transaction, for the savepoints to sit inside | Savepoints |
| `SAVEPOINT station` and `ROLLBACK TO station` | undoing one station's rows and keeping the others | Savepoints |
| the reader seeing nothing before the end | other connections see committed changes only | commit, and what another connection sees |
| one broken station, and the others kept | all or nothing, where it matters, and no further | All or nothing: replacing a day of readings |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/08-transactions-solutions.ipynb).

**1.** Insert an Oslo reading of -3.0 for 00:00 on 1 January 2026, and print `in_transaction` and
what a second connection counts for that day, both before and after `commit`.


In [8]:
# your code here


**2.** Delete every Tromso reading from 1 January, show that they are gone, roll back, and show that
all 24 are back.


In [9]:
# your code here


**3.** In a `with` block, insert a Bergen reading for 00:00 on 2 January 2026, then raise an
exception before a second insert, catch it outside the block, and show that no reading for that day
was saved.


In [10]:
# your code here


**4.** In one transaction opened with `BEGIN`, insert three Bergen readings for 3 January 2026 with a
savepoint before the third, roll back to the savepoint, commit, and show that two were saved.


In [11]:
# your code here


**5.** Show that `executescript` commits a pending insert: print `in_transaction` before and after
it, then roll back, and count the row from a second connection.


In [12]:
# your code here


**6.** Write `replace_reading(conn, station, hour, celsius)`, which deletes a station's reading for
an hour and inserts the new value in one transaction, raising `ValueError` after the delete when the
new value is outside -60 to 60 degrees. Show that a failed replacement leaves the old reading in
place.


In [13]:
# your code here


## Common errors

### No error, and no rows: a connection closed before its commit


In [14]:
careless = sqlite3.connect(DATABASE)
careless.execute(INSERT_READING,
                 (station_ids["Kirkenes"], "2025-12-04T00:00", -9.0))
print("before closing, it sees:", count_readings(careless, "Kirkenes", "2025-12-04"))
careless.close()

careless = sqlite3.connect(DATABASE)
print("after connecting again: ", count_readings(careless, "Kirkenes", "2025-12-04"))
careless.close()


before closing, it sees: 1
after connecting again:  0


The insert opened a transaction, the connection read its own row back, and `close()` ended the
transaction without committing it, which threw the row away. Nothing warned about it: under sqlite3's
default transaction control, closing a connection runs no commit and says nothing. Commit before
closing, or write the change in a `with` block, which commits as it ends:


In [15]:
careful = sqlite3.connect(DATABASE)
with careful:
    careful.execute(INSERT_READING,
                    (station_ids["Kirkenes"], "2025-12-04T00:00", -9.0))
careful.close()

careful = sqlite3.connect(DATABASE)
print("after connecting again:", count_readings(careful, "Kirkenes", "2025-12-04"))
careful.close()


after connecting again: 1


### sqlite3.OperationalError: cannot start a transaction within a transaction


In [16]:
writer.execute(INSERT_READING,
               (station_ids["Kirkenes"], "2025-12-04T01:00", -9.1))
writer.execute("BEGIN")


OperationalError: cannot start a transaction within a transaction

The insert opened a transaction before `BEGIN` ran, and SQLite transactions do not nest. Nothing in
the code shows that open transaction, which is what makes this error confusing: the only clue is
`in_transaction`. End the open transaction first, with a commit or a rollback, or mark the step with
a savepoint instead of a new transaction:


In [17]:
print("in a transaction:", writer.in_transaction)
writer.commit()

writer.execute("BEGIN")
writer.execute(INSERT_READING,
               (station_ids["Kirkenes"], "2025-12-04T02:00", -9.2))
writer.commit()
print("Kirkenes readings on 4 December:", count_readings(reader, "Kirkenes", "2025-12-04"))


in a transaction: True
Kirkenes readings on 4 December: 3


### No error, and nothing left to roll back: executescript committed first


In [18]:
writer.execute(INSERT_READING,
               (station_ids["Kirkenes"], "2025-12-04T03:00", 93.0))   # a mistyped reading, to be rolled back
print("in a transaction before executescript:", writer.in_transaction)

writer.executescript("CREATE TABLE IF NOT EXISTS audit (note TEXT NOT NULL);")
print("in a transaction after executescript: ", writer.in_transaction)

writer.rollback()
mistyped = "SELECT hour, celsius FROM readings WHERE station_id = ? AND hour = ?"
print("after rollback:", reader.execute(mistyped, (station_ids["Kirkenes"], "2025-12-04T03:00")).fetchall())


in a transaction before executescript: True
in a transaction after executescript:  False
after rollback: [('2025-12-04T03:00', 93.0)]


`executescript` commits any open transaction before it runs its script, so the mistyped reading was
saved as the table was created, and the `rollback` that was meant to remove it found no transaction
to undo. The **Changing a Schema** notebook relies on this same commit. Finish the open transaction
before calling `executescript`, deciding on purpose whether to keep its changes, and since this
reading was committed, remove it with a change of its own:


In [19]:
with writer:
    writer.execute("DELETE FROM readings WHERE station_id = ? AND hour = ?", (station_ids["Kirkenes"], "2025-12-04T03:00"))

writer.execute(INSERT_READING,
               (station_ids["Kirkenes"], "2025-12-04T03:00", 93.0))
writer.rollback()                                           # decided first: this reading is not kept
writer.executescript("CREATE TABLE IF NOT EXISTS audit (note TEXT NOT NULL);")
print("the mistyped reading:", reader.execute(mistyped, (station_ids["Kirkenes"], "2025-12-04T03:00")).fetchall())


the mistyped reading: []


### No error, and half a change saved: an exception caught, then a commit


In [20]:
def replace_day_carelessly(conn, station, day, estimates):
    """Replace a day's readings with estimates, with no transaction of its own, which is the mistake."""
    conn.execute("DELETE FROM readings WHERE station_id = ? AND hour LIKE ?", (station_ids[station], f"{day}%"))
    for hour in range(24):
        conn.execute(INSERT_READING,
                     (station_ids[station], f"{day}T{hour:02d}:00", estimates[hour]))


try:
    replace_day_carelessly(writer, "Bergen", "2025-06-21", {hour: 15.0 for hour in range(12)})   # only half a day
except KeyError as error:
    print("caught: no estimate for hour", error)

with writer:                                               # later, and unrelated
    writer.execute(INSERT_READING,
                   (station_ids["Kirkenes"], "2025-12-04T04:00", -9.4))
print("Bergen's readings on 21 June:", count_readings(reader, "Bergen", "2025-06-21"))


caught: no estimate for hour 12
Bergen's readings on 21 June: 12


The delete and twelve inserts were still pending when `KeyError` reached the `except`, and catching
an exception does not roll anything back. The next commit, from a `with` block that had nothing to
do with Bergen, saved everything pending on the connection, so Bergen's midsummer day now holds 12
readings where it held 24. Give the function its own `with` block, so the exception rolls its
changes back before any caller sees it. The day's real readings come back from `year_of_readings`:


In [21]:
with writer:
    writer.execute("DELETE FROM readings WHERE station_id = ? AND hour LIKE ?", (station_ids["Bergen"], "2025-06-21%"))
    writer.executemany(INSERT_READING,
                       [(station_ids[station], hour, celsius) for station, hour, celsius in year_of_readings()
                        if station == "Bergen" and hour.startswith("2025-06-21")])


def replace_day(conn, station, day, estimates):
    """Replace a day's readings with estimates in one transaction, so a failure leaves the day as it was."""
    with conn:
        replace_day_carelessly(conn, station, day, estimates)


try:
    replace_day(writer, "Bergen", "2025-06-21", {hour: 15.0 for hour in range(12)})
except KeyError as error:
    print("caught: no estimate for hour", error)
print("Bergen's readings on 21 June:", count_readings(reader, "Bergen", "2025-06-21"))


caught: no estimate for hour 12
Bergen's readings on 21 June: 24


### No error, and the outer block's reading gone too: a with block inside another


In [22]:
def add_reading(conn, station, hour, celsius):
    """Insert one reading in a with block of its own, which is the mistake when a caller has one too."""
    with conn:
        if not -60 <= celsius <= 60:
            raise ValueError(f"{celsius} is not a plausible temperature")
        conn.execute(INSERT_READING,
                     (station_ids[station], hour, celsius))


with writer:
    writer.execute(INSERT_READING,
                   (station_ids["Kirkenes"], "2025-12-05T00:00", -9.3))
    try:
        add_reading(writer, "Kirkenes", "2025-12-05T01:00", 93.0)
    except ValueError as error:
        print("skipped:", error)
    add_reading(writer, "Kirkenes", "2025-12-05T02:00", -9.6)

print(reader.execute("SELECT hour, celsius FROM readings WHERE station_id = ? AND hour LIKE ? ORDER BY hour",
                     (station_ids["Kirkenes"], "2025-12-05%")).fetchall())


skipped: 93.0 is not a plausible temperature
[('2025-12-05T02:00', -9.6)]


The outer block inserted 00:00, and only 02:00 was saved. `add_reading`'s own `with` block rolled
back when it raised, and a rollback throws away the whole transaction, including the outer block's
pending insert, since there is only ever one transaction and no inner one to undo alone. Had the
inner block finished instead, it would have committed the outer block's work early. A step inside a
transaction marks itself with a savepoint, which undoes only that step:


In [23]:
def add_reading(conn, station, hour, celsius):
    """Insert one reading inside a savepoint, which undoes only this reading if it fails."""
    conn.execute("SAVEPOINT reading")
    try:
        if not -60 <= celsius <= 60:
            raise ValueError(f"{celsius} is not a plausible temperature")
        conn.execute(INSERT_READING,
                     (station_ids[station], hour, celsius))
    except ValueError:
        conn.execute("ROLLBACK TO reading")
        raise
    finally:
        conn.execute("RELEASE reading")


with writer:
    writer.execute("BEGIN")
    writer.execute(INSERT_READING,
                   (station_ids["Kirkenes"], "2025-12-06T00:00", -9.3))
    try:
        add_reading(writer, "Kirkenes", "2025-12-06T01:00", 93.0)
    except ValueError as error:
        print("skipped:", error)
    add_reading(writer, "Kirkenes", "2025-12-06T02:00", -9.6)

print(reader.execute("SELECT hour, celsius FROM readings WHERE station_id = ? AND hour LIKE ? ORDER BY hour",
                     (station_ids["Kirkenes"], "2025-12-06%")).fetchall())


skipped: 93.0 is not a plausible temperature
[('2025-12-06T00:00', -9.3), ('2025-12-06T02:00', -9.6)]


### sqlite3.OperationalError: cannot commit - no transaction is active


In [24]:
with writer:
    writer.execute(INSERT_READING,
                   (station_ids["Kirkenes"], "2025-12-07T00:00", -9.8))
writer.execute("COMMIT")


OperationalError: cannot commit - no transaction is active

The `with` block had already committed, so the SQL statement `COMMIT` found no transaction to
commit, and SQLite refuses that. The method `commit()` is the forgiving one, doing nothing when no
transaction is open. Leave the committing to the `with` block, or call `conn.commit()`, not the SQL
statement:


In [25]:
writer.commit()
print("in a transaction:", writer.in_transaction)
print("Kirkenes readings on 7 December:", count_readings(reader, "Kirkenes", "2025-12-07"))

reader.close()
writer.close()


in a transaction: False
Kirkenes readings on 7 December: 1


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
database in it:


In [26]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A transaction applies its changes completely or not at all: `commit` makes them permanent and
  visible to every connection, and `rollback` throws them away.
- sqlite3 opens a transaction before an `INSERT`, `UPDATE`, `DELETE` or `REPLACE`, and
  `in_transaction` says whether one is open. Nothing commits it but `commit`, a `with` block, or a
  statement that commits first, such as `executescript`.
- Closing a connection without a commit throws its open transaction away.
- `with conn:` commits a block that finishes and rolls back one that raises, and neither opens a
  transaction nor closes the connection.
- Transactions do not nest: `BEGIN` inside one is an error, and a `with conn:` inside another ends
  the outer transaction. A savepoint marks a step, and `ROLLBACK TO` undoes only that step.
- Catching an exception rolls nothing back, so a change that can fail belongs in its own `with`
  block.


## What is next

The **autocommit and isolation_level** notebook looks at the rules this notebook took on trust: when
exactly sqlite3 opens a transaction, the `CREATE TABLE` that a rollback could not undo, and the
`autocommit` setting that changes both.


---

&#8592; **Previous:** [Adapters and Converters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/07-adapters-and-converters.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
